In [2]:
# NVIDIA (CUDA)
import torch

print(f'cuda: {torch.cuda.is_available()}')  # True/False

# Apple Silicon (MPS — Metal Performance Shaders)
import torch

print(f'mps: {torch.backends.mps.is_available()}')  # True/False
print(f'mps built: {torch.backends.mps.is_built()}')  # Checks if PyTorch was built with MPS support

cuda: False
mps: True
mps built: True


In [3]:
import torch
torch.cuda.is_available()

False

In [7]:
import pickle

In [32]:
with open("checkpoints/train_bpe_tinystories.pkl", "rb") as f:
    data = pickle.load(f)
    vocab, merges = data["vocab"], data["merges"]

In [33]:
max(vocab.values(), key= len)

b' accomplishment'

In [36]:
merges[-20:]

[(b' slice', b's'),
 (b' sle', b'igh'),
 (b' sl', b'ime'),
 (b' sign', b'al'),
 (b' sick', b'ness'),
 (b' shy', b'ly'),
 (b' serv', b'ants'),
 (b' serious', b'ly'),
 (b' ser', b'ving'),
 (b' secret', b'ly'),
 (b' se', b'ction'),
 (b' sculpture', b's'),
 (b' scru', b'bbing'),
 (b' scr', b'unch'),
 (b' scr', b'ambled'),
 (b' scoop', b's'),
 (b' scen', b'e'),
 (b' scare', b's'),
 (b' sc', b'owl'),
 (b' sc', b'ientist')]

In [5]:
import regex as re
text = "I want to go home. It's my time to go home in the evening. I want to go home. new-Horizon"
pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
print(re.findall(pattern, text))

['I', ' want', ' to', ' go', ' home', '.', ' It', "'s", ' my', ' time', ' to', ' go', ' home', ' in', ' the', ' evening', '.', ' I', ' want', ' to', ' go', ' home', '.', ' new', '-', 'Horizon']


In [7]:
import pickle
with open("../checkpoints/bpe_tinystories.pkl", "rb") as f:
    data = pickle.load(f)
    vocab, merges, pretokenization_pattern = data["vocab"], data["merges"], data["pretokenization_pattern"]
print(len(vocab), len(merges), pretokenization_pattern)

10000 9743 r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


In [ ]:
special_tokens = [b"<endoftext>", b"<pad>", b"<unk>", b"<mask>"]
text = "I want to go home. It's my time to go home in the evening.<endoftext> I want to go home. new-Horizon"


In [ ]:
input_path = "../data/TinyStoriesV2-GPT4-train-10k.txt"
vocab_size = 1000
special_tokens = [b"<|endoftext|>", b"<|unknown|>"]
split_special_token = b"<|endoftext|>"
desired_num_chunks = 1
pretokenization_pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


from pretokenization_example import find_chunk_boundaries
import regex as re

file = open(input_path, "rb")
count = 0
vocab = {}
merges = []

# Addding all bytes to the vocabulary
for i in range(256):
    vocab[count] = bytes([i])
    count += 1

# Adding special tokens to the vocabulary
for token in special_tokens:
    vocab[count] = token
    count += 1

# Parallelizing pre-tokenization
boundaries = find_chunk_boundaries(file=file, desired_num_chunks=desired_num_chunks, split_special_token=split_special_token)
for i in range(len(boundaries) - 1):
    start, end = boundaries[i], boundaries[i + 1]
    file.seek(start)
    chunk = file.read(end - start)

    # Removing special tokens before pre-tokenization
    pattern = b"|".join(re.escape(tok) for tok in special_tokens)
    docs = re.split(pattern, chunk)

    # Pretokenization
    pretokens = {}
    for doc in docs:
        for x in re.finditer(pretokenization_pattern.encode("utf-8"), doc):
            pretoken = tuple([bytes([b]) for b in x.group(0)])
            pretokens[pretoken] = pretokens.get(pretoken, 0) + 1

    # Merging
    while len(vocab) < vocab_size:
        # Find the most frequent pair of tokens
        pairs = {}
        for pretoken, freq in pretokens.items():
            for i in range(len(pretoken) - 1):
                pair = (pretoken[i], pretoken[i+1])
                pairs[pair] = pairs.get(pair, 0) + freq

        if not pairs:
            break

        most_frequent_pair = max(pairs, key=lambda pair: (pairs[pair], pair))
        new_token = most_frequent_pair[0] + most_frequent_pair[1]
        # Add the new token to the vocabulary
        vocab[count] = new_token
        count += 1
        merges.append(most_frequent_pair)

        # Update pretokens with the new token
        new_pretokens = {}
        for pretoken, freq in pretokens.items():
            new_pretoken = []
            i = 0
            while i < len(pretoken):
                if i < len(pretoken) - 1 and (pretoken[i], pretoken[i+1]) == most_frequent_pair:
                    new_pretoken.append(new_token)
                    i += 2
                else:
                    new_pretoken.append(pretoken[i])
                    i += 1
            new_pretokens[tuple(new_pretoken)] = new_pretokens.get(tuple(new_pretoken), 0) + freq

        pretokens = new_pretokens
    

In [114]:
from bpe_training import train_bpe
vocab, merges = train_bpe(
        input_path = "../data/TinyStoriesV2-GPT4-valid.txt",
        vocab_size = 10000,
        special_tokens = [b"<|endoftext|>"],#, b"<|unknown|>"],
        split_special_token = b"<|endoftext|>",
        desired_num_chunks = 1,
        pretokenization_pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""",
    )

In [121]:
for i in range(9800, 10000):
    print(vocab[i], end=" ")

b'eil' b'ece' b'eat' b'ean' b'des' b'daughter' b'cutter' b'colored' b'coat' b'cellent' b'bbling' b'bs' b'bounce' b'boom' b'bling' b'bang' b'azel' b'aylor' b'aura' b'athy' b'atelyn' b'atform' b'atal' b'atalie' b'arted' b'ars' b'applaud' b'aptain' b'aphne' b'angerous' b'ancelot' b'anation' b'ample' b'ambled' b'alerious' b'Yesterday' b'Years' b'Whist' b'Wo' b'Wisdom' b'Whe' b'Wally' b'Violet' b'Trixie' b'Ten' b'Stella' b'Spin' b'Spark' b'Sweet' b'Squ' b'Sc' b'Ros' b'Pippa' b'Pretty' b'Playing' b'Play' b'Pizza' b'Pin' b'Oxygen' b'Move' b'Mittens' b'Mila' b'Michael' b'Many' b'Mabel' b'Lion' b'Lauren' b'Laura' b'Kelsey' b'Kay' b'Jacky' b'Jilly' b'Izzy' b'Hannah' b'Grow' b'Frog' b'Finn' b'Free' b'Feeling' b'Ever' b'Elephant' b'Eddie' b'Daniel' b'Dino' b'Dare' b'Check' b'Catty' b'Confused' b'Celina' b'Brownie' b'Birds' b'Bu' b'Books' b'Ball' b'Auntie' b'Ahh' b'Ant' b'AT' b'?!' b'.\xe2\x80\x99' b"!'." b"!''" b' zips' b' zebras' b' yielding' b' wriggled' b' worst' b' woodcutter' b' winking' b' w

In [ ]:
def pretokenize(text):

In [31]:
# from pretokenization_example import find_chunk_boundaries
# with open(input_path, "rb") as f:
#     num_processes = 4
#     boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

#     # The following is a serial implementation, but you can parallelize this
#     # by sending each start/end pair to a set of processes.
#     for start, end in zip(boundaries[:-1], boundaries[1:]):
#         f.seek(start)
#         chunk = f.read(end - start).decode("utf-8", errors="ignore")
#         break
#         # Run pre-tokenization on your chunk and store the counts for each pre-token